In [2]:
import os
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
ORIGINAL_DATA_PATH = "../original-data"
PROCESSED_DATA_PATH = "../processed-data"

In [6]:
print("1. Đọc dữ liệu đã tiền xử lý...")
df = pd.read_csv(f'{PROCESSED_DATA_PATH}/final_training_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Chuyển đổi các cột chuỗi (nếu còn) sang Category cho LightGBM
object_cols = [col for col in df.select_dtypes(include=['object']).columns if col not in ['Date', 'Split']]
for col in object_cols:
    df[col] = df[col].astype('category')

print("\n2. Phân tách dữ liệu (Time Series Split)...")
# Lấy tập Train
train_full = df[df['Split'] == 'Train'].copy()

# Tách Validation Set: Huấn luyện trên dữ liệu trước 2022, Validate trên năm 2022
train_set = train_full[train_full['year'] < 2022].copy()
val_set = train_full[train_full['year'] == 2022].copy()

print(f"Số dòng Train: {len(train_set)}, Validation: {len(val_set)}")

print("\n3. Xác định Features và Target...")
cols_to_drop = ['Date', 'Split', 'Revenue', 'COGS']
features = [col for col in train_full.columns if col not in cols_to_drop]

X_train = train_set[features]
X_val = val_set[features]

y_train_rev = train_set['Revenue']
y_val_rev = val_set['Revenue']

y_train_cogs = train_set['COGS']
y_val_cogs = val_set['COGS']

# ==========================================
# 4. THIẾT LẬP HYPERPARAMETERS (Tối ưu từ forecast.py)
# ==========================================
# Tăng n_estimators lên 5000, giảm learning_rate, tăng num_leaves để chống Underfitting
lgb_params = {
    'objective': 'regression', 'metric': 'mae', 
    'learning_rate': 0.02, 'num_leaves': 127, 'min_child_samples': 15, 
    'subsample': 0.8, 'colsample_bytree': 0.7, 
    'reg_alpha': 0.1, 'reg_lambda': 0.5, 
    'n_estimators': 5000, 'random_state': SEED, 'n_jobs': -1
}

xgb_params = {
    'objective': 'reg:squarederror', 'eval_metric': 'mae', 
    'learning_rate': 0.02, 'max_depth': 7, 
    'subsample': 0.8, 'colsample_bytree': 0.7, 
    'alpha': 0.1, 'lambda': 0.5, 
    'n_estimators': 5000, 'random_state': SEED, 'n_jobs': -1,
    'early_stopping_rounds': 50
}

# ==========================================
# 5. HUẤN LUYỆN & ĐÁNH GIÁ (REVENUE) - ENSEMBLE
# ==========================================
print("\n--- Bắt đầu huấn luyện mô hình REVENUE ---")
print("-> Training LightGBM...")
model_lgb_rev = lgb.LGBMRegressor(**lgb_params)
model_lgb_rev.fit(X_train, y_train_rev, eval_set=[(X_val, y_val_rev)], callbacks=[lgb.early_stopping(50, verbose=False)])

print("-> Training XGBoost...")
model_xgb_rev = xgb.XGBRegressor(**xgb_params, enable_categorical=True)
model_xgb_rev.fit(X_train, y_train_rev, eval_set=[(X_val, y_val_rev)], verbose=False)

# Dự báo Ensemble (Trung bình cộng 2 mô hình)
pred_val_rev = 0.5 * model_lgb_rev.predict(X_val) + 0.5 * model_xgb_rev.predict(X_val)

print(f"\n[Kết quả Ensemble Validation - REVENUE]")
print(f"MAE  = {mean_absolute_error(y_val_rev, pred_val_rev):,.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_val_rev, pred_val_rev)):,.2f}")
print(f"R^2  = {r2_score(y_val_rev, pred_val_rev):.4f}")

# ==========================================
# 6. HUẤN LUYỆN & ĐÁNH GIÁ (COGS) - ENSEMBLE
# ==========================================
print("\n--- Bắt đầu huấn luyện mô hình COGS ---")
print("-> Training LightGBM...")
model_lgb_cogs = lgb.LGBMRegressor(**lgb_params)
model_lgb_cogs.fit(X_train, y_train_cogs, eval_set=[(X_val, y_val_cogs)], callbacks=[lgb.early_stopping(50, verbose=False)])

print("-> Training XGBoost...")
model_xgb_cogs = xgb.XGBRegressor(**xgb_params, enable_categorical=True)
model_xgb_cogs.fit(X_train, y_train_cogs, eval_set=[(X_val, y_val_cogs)], verbose=False)

pred_val_cogs = 0.5 * model_lgb_cogs.predict(X_val) + 0.5 * model_xgb_cogs.predict(X_val)

print(f"\n[Kết quả Ensemble Validation - COGS]")
print(f"MAE  = {mean_absolute_error(y_val_cogs, pred_val_cogs):,.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_val_cogs, pred_val_cogs)):,.2f}")
print(f"R^2  = {r2_score(y_val_cogs, pred_val_cogs):.4f}")

# ==========================================
# 7. KHẢ NĂNG GIẢI THÍCH (SHAP ANALYSIS)
# ==========================================
print("\n--- GIAI ĐOẠN PHÂN TÍCH SHAP (YÊU CẦU BẮT BUỘC) ---")
print("Đang tính toán SHAP values từ mô hình LightGBM Revenue...")

# Chỉ cần dùng LightGBM để phân tích SHAP là đủ đại diện
explainer = shap.TreeExplainer(model_lgb_rev)
shap_values = explainer.shap_values(X_val)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_val, plot_type="bar", show=False)
plt.title("Mức độ quan trọng của các đặc trưng (Feature Importance)")
plt.tight_layout()
plt.savefig('shap_importance_bar.png', dpi=300)
plt.close()

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_val, show=False)
plt.title("Tác động của đặc trưng lên Doanh thu (SHAP Beeswarm)")
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', dpi=300)
plt.close()

print("Hoàn tất! Đã lưu 2 biểu đồ phân tích SHAP.")

1. Đọc dữ liệu đã tiền xử lý...

2. Phân tách dữ liệu (Time Series Split)...
Số dòng Train: 3103, Validation: 365

3. Xác định Features và Target...

--- Bắt đầu huấn luyện mô hình REVENUE ---
-> Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000535 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2663
[LightGBM] [Info] Number of data points in the train set: 3103, number of used features: 24
[LightGBM] [Info] Start training from score 4365694.871304
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
-> Training XGBoost...

[Kết quả Ensemble Validation - REVENUE]
MAE  = 547,785.17
RMSE = 775,519.49
R^2  = 0.7853

--- Bắt đầu huấn luyện mô hình COGS ---
-> Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000323 seconds.
You